# Config-Driven Training Workspace

This notebook is intentionally thin. Load a YAML config from `configs/`, inspect what it resolves to, visualize a sample, and run the same training path used by the CLI.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == 'model' else cwd
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from model.config import load_training_config
from model.train import (
    build_dataset_from_config,
    build_model_from_config,
    load_checkpoint,
    predict_frames,
    run_training_from_config,
)

print('repo_root:', repo_root)
print('cuda_available:', torch.cuda.is_available())

In [ ]:
CONFIG_PATH = repo_root / 'configs/coord_to_image_unet_all_current_data_v1.yaml'
OVERRIDES = []

config = load_training_config(CONFIG_PATH, overrides=OVERRIDES)
for line in config.summary_lines():
    print(line)

if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print('gpu:', torch.cuda.get_device_name(0))
    print('free_gb:', round(free_bytes / 1024**3, 2))
    print('total_gb:', round(total_bytes / 1024**3, 2))

In [ ]:
dataset = build_dataset_from_config(config)
print('total dataset size:', len(dataset))
print('split artifact path:', config.split_artifact_path)
print('resolved config artifact path:', config.resolved_config_path)

coords, frame = dataset[0]
image = frame.permute(1, 2, 0).cpu().numpy()
x = coords[0].item() * image.shape[1]
y = coords[1].item() * image.shape[0]

plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.scatter([x], [y], c='red', s=60)
plt.title(f'Training target example from {config.version}')
plt.axis('off')
plt.show()

In [ ]:
result = run_training_from_config(config)
checkpoint_path = result.final_checkpoint_path or result.best_checkpoint_path
print('device:', result.device)
print('resumed_from_checkpoint:', result.resumed_from_checkpoint)
print('split_artifact_path:', result.split_artifact_path)
print('resolved_config_path:', result.resolved_config_path)
print('final_checkpoint_path:', result.final_checkpoint_path)
print('best_checkpoint_path:', result.best_checkpoint_path)
print('wandb_run_id:', result.wandb_run_id)
print('wandb_run_name:', result.wandb_run_name)
print('wandb_run_url:', result.wandb_run_url)
print('checkpoint_path_for_reload:', checkpoint_path)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(result.history.train_losses, label='train')
plt.plot(result.history.val_losses, label='val')
plt.plot(result.history.test_losses, label='test')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title(f'Training curves for {config.version}')
plt.legend()
plt.show()

In [ ]:
reloaded_model = build_model_from_config(config).to(result.device)
reloaded_model, reloaded_history, reloaded_extra = load_checkpoint(
    checkpoint_path,
    reloaded_model,
    device=result.device,
)
print('reloaded_history:', reloaded_history)
print('reloaded_extra:', reloaded_extra)

eval_loader = result.test_loader or result.val_loader
test_coords, test_frames = next(iter(eval_loader))
num_examples = min(4, test_coords.shape[0])
pred_frames = predict_frames(reloaded_model, test_coords[:num_examples], device=result.device)
target_frames = test_frames[:num_examples]

fig, axes = plt.subplots(num_examples, 2, figsize=(8, 4 * num_examples))
if num_examples == 1:
    axes = [axes]
for row in range(num_examples):
    target = target_frames[row].permute(1, 2, 0).cpu().numpy()
    pred = pred_frames[row].permute(1, 2, 0).cpu().numpy()
    axes[row][0].imshow(target)
    axes[row][0].set_title(f'target {row}')
    axes[row][0].axis('off')
    axes[row][1].imshow(pred)
    axes[row][1].set_title(f'prediction {row}')
    axes[row][1].axis('off')

plt.tight_layout()
plt.show()